In [2]:
# cell no - 2
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torchvision.models as models
from torch.utils.data import DataLoader

import numpy as np
import matplotlib.pyplot as plt
import os

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap

# ===== CONFIG =====
MODEL_NAME = "resnet50"   # change this ONLY (resnet50, vgg16, mobilenet_v2)

BASE_PATH = "/content/drive/MyDrive/4-2/Deep Learning/Assignments/Assignment-2"

FACE_DATA_PATH = f"{BASE_PATH}/dataset/Five_faces_split"
MY_FACE_PATH   = f"{BASE_PATH}/dataset/my_faces"

RESULTS_PATH   = f"{BASE_PATH}/results"

BATCH_SIZE = 32
EPOCHS = 5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [3]:
# cell no - 3
def get_transforms():
    return transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

In [4]:
# cell no - 4
def load_face_datasets(transform):
    train = datasets.ImageFolder(os.path.join(FACE_DATA_PATH, "train"), transform=transform)
    val   = datasets.ImageFolder(os.path.join(FACE_DATA_PATH, "val"), transform=transform)

    train_loader = DataLoader(train, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val, batch_size=BATCH_SIZE, shuffle=False)

    return train_loader, val_loader, train.classes


def load_my_faces(transform):
    dataset = datasets.ImageFolder(MY_FACE_PATH, transform=transform)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)
    return loader, dataset.targets

In [5]:
# cell no - 5 (MODIFIED)
from torchvision.models import resnet50, vgg16, mobilenet_v2
from torchvision.models import ResNet50_Weights, VGG16_Weights, MobileNet_V2_Weights

def get_model(model_name, num_classes=None, pretrained=True):

    if model_name == "resnet50":
        weights = ResNet50_Weights.DEFAULT if pretrained else None
        model = resnet50(weights=weights)
        in_features = model.fc.in_features
        if num_classes:
            model.fc = nn.Linear(in_features, num_classes)
        feature_extractor = nn.Sequential(*list(model.children())[:-1])

    elif model_name == "vgg16":
        weights = VGG16_Weights.DEFAULT if pretrained else None
        model = vgg16(weights=weights)
        in_features = model.classifier[-1].in_features
        if num_classes:
            model.classifier[-1] = nn.Linear(in_features, num_classes)
        feature_extractor = nn.Sequential(*list(model.features), nn.AdaptiveAvgPool2d((7,7)), nn.Flatten())

    elif model_name == "mobilenet_v2":
        weights = MobileNet_V2_Weights.DEFAULT if pretrained else None
        model = mobilenet_v2(weights=weights)
        in_features = model.classifier[-1].in_features
        if num_classes:
            model.classifier[-1] = nn.Linear(in_features, num_classes)
        feature_extractor = nn.Sequential(model.features, nn.AdaptiveAvgPool2d((1,1)), nn.Flatten())

    else:
        raise ValueError("Unsupported model")

    return model.to(device), feature_extractor.to(device)

In [6]:
# cell no - 6
def extract_features(model, loader):
    model.eval()
    features = []
    labels = []

    with torch.no_grad():
        for imgs, lbls in loader:
            imgs = imgs.to(device)
            output = model(imgs)
            output = output.view(output.size(0), -1)

            features.append(output.cpu().numpy())
            labels.append(lbls.numpy())

    return np.vstack(features), np.hstack(labels)

In [7]:
# cell no - 7
def train_model(model, train_loader, epochs):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    model.train()

    for epoch in range(epochs):
        total_loss = 0

        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

In [8]:
# cell no - 8 (MODIFIED)
def reduce_dimensions(features):
    results = {}

    results['pca'] = PCA(n_components=2).fit_transform(features)

    n_samples = features.shape[0]
    perplexity = max(2, min(30, n_samples // 3))  # FIX

    results['tsne'] = TSNE(n_components=2, perplexity=perplexity).fit_transform(features)
    results['umap'] = umap.UMAP(n_components=2).fit_transform(features)

    return results

In [9]:
# cell no - 9
def save_plot(embedding, labels, title, save_path):
    plt.figure()
    plt.scatter(embedding[:,0], embedding[:,1], c=labels)
    plt.title(title)
    plt.colorbar()

    plt.savefig(save_path)
    plt.close()

In [10]:
# cell no - 10
def main():
    transform = get_transforms()

    # Load data
    train_loader, val_loader, classes = load_face_datasets(transform)
    my_loader, my_labels = load_my_faces(transform)

    # Create results folder
    model_result_path = os.path.join(RESULTS_PATH, MODEL_NAME)
    os.makedirs(model_result_path, exist_ok=True)

    # ===== BEFORE FINE-TUNING =====
    base_model, feature_extractor = get_model(MODEL_NAME, pretrained=True)

    features_before, labels = extract_features(feature_extractor, my_loader)
    reduced_before = reduce_dimensions(features_before)

    for method, emb in reduced_before.items():
        save_plot(
            emb, labels,
            f"{MODEL_NAME} BEFORE {method}",
            os.path.join(model_result_path, f"before_{method}.png")
        )

    # ===== AFTER FINE-TUNING =====
    model_ft, _ = get_model(MODEL_NAME, num_classes=len(classes), pretrained=True)

    train_model(model_ft, train_loader, EPOCHS)

    # Extract features after FT

    feature_extractor_ft = nn.Sequential(*list(model_ft.children())[:-1])
    feature_extractor_ft = feature_extractor_ft.to(device)
    feature_extractor_ft.eval()

    features_after, labels = extract_features(feature_extractor_ft, my_loader)
    reduced_after = reduce_dimensions(features_after)

    for method, emb in reduced_after.items():
        save_plot(
            emb, labels,
            f"{MODEL_NAME} AFTER {method}",
            os.path.join(model_result_path, f"after_{method}.png")
        )

    print("Done. Results saved at:", model_result_path)

In [11]:
# cell no - 11
main()

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 206MB/s]


Epoch 1, Loss: 23.0616
Epoch 2, Loss: 9.2856
Epoch 3, Loss: 3.2014
Epoch 4, Loss: 0.7376
Epoch 5, Loss: 0.4238
Done. Results saved at: /content/drive/MyDrive/4-2/Deep Learning/Assignments/Assignment-2/results/resnet50
